# Farneback

Mac: `small-movts` Micromamba env

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

video_fname = "28-MAY-2025_00h04m33.969s.mp4"
data_dir = Path("data/farneback/subj1_cam1")
video_path = data_dir / video_fname

In [ ]:
# ROI: x, y, width, height (adjust to your video)
x, y, w, h = 200, 100, 120, 80

cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)

ret, frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read video")

In [ ]:
# 10m 38s on Mac for 5m-long video
prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
# prev = prev[y:y+h, x:x+w]

times = []
intensities = []
frame_idx = 1

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # curr = gray[y:y+h, x:x+w]

    flow = cv2.calcOpticalFlowFarneback(
        prev, gray, None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0,
    )

    magnitude = np.linalg.norm(flow, axis=2)

    # Ignore tiny displacements likely caused by noise
    magnitude[magnitude < 0.3] = 0

    # Mean motion intensity, in pixels per second
    intensity = magnitude.mean() * fps

    times.append(frame_idx / fps)
    intensities.append(intensity)

    prev = gray # curr
    frame_idx += 1

cap.release()

In [ ]:
plt.plot(times, intensities)
plt.xlabel("Time (s)")
plt.ylabel("Motion intensity (pixels/s)")
plt.tight_layout()
plt.savefig(video_path.parent / f"{video_path.stem}_farneback_intensity.pdf")

In [ ]:
from typing import Iterable

def save_txt(arr: Iterable, out_path: Path) -> None:
    np.savetxt(str(out_path), np.array(arr), fmt="%.6f")

times_path = data_dir / f"{video_path.stem}_times.txt"
intensities_path = data_dir / f"{video_path.stem}_intensities.txt"

In [ ]:
# save_txt(times, times_path)
# save_txt(intensities, intensities_path)

In [ ]:
from typing import Final

times_np = np.loadtxt(times_path) # times in seconds, 1D array
intensities_np = np.loadtxt(intensities_path)

# calc sliding average
periods = (# 1, 2, 5,
           8, 12
           )  # in seconds
time_delta: Final[float] = times_np[1] - times_np[0]

for p in periods:
    n_points = int(p // time_delta)
    print(n_points)

    averages = []
    for i in range(len(times_np) - n_points):
        ave = np.average(intensities_np[i:i + n_points])
        averages.append(ave)
    plt.figure()
    plt.plot(times_np[n_points:], averages)

# EEG analysis

Mac: `femba` Micromamba env

In [ ]:
import mne
from src.video_utils import parse_timestamp_from_filename

time_str_video, sec_start_video = parse_timestamp_from_filename(video_fname)
time_str

In [ ]:
eeg_fname = "27-MAY-2025_22h19m28.399s.edf"
time_str_eeg, sec_start_eeg = parse_timestamp_from_filename(eeg_fname)
time_str_eeg

In [ ]:
video_offset_sec = sec_start_video - sec_start_eeg
video_offset_sec

In [ ]:
from pathlib import Path

eeg_path = Path("data/farneback") / eeg_fname
raw = mne.io.read_raw_edf(str(eeg_path))

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch


duration_sec = 5 * 60
window_sec = 4.0
step_sec = 1.0
alpha_band = (8.0, 13.0)
delta_band = (0.5, 4.0)


# Select EEG channels and crop the fragment
# eeg_picks = mne.pick_types(raw.info, eeg=True, exclude="bads")
raw_fragment = raw.copy().crop(
    tmin=video_offset_sec,
    tmax=video_offset_sec + duration_sec,
    include_tmax=False,
    # picks=eeg_picks,
)
raw_fragment.drop_channels(['ECG  ECG'])

data = raw_fragment.get_data()  # (n_channels, n_samples)
sfreq = raw_fragment.info["sfreq"]

# Compute average band power in sliding windows
window_samples = int(window_sec * sfreq)
step_samples = int(step_sec * sfreq)

times = []
alpha_power = []
delta_power = []

for start in range(0, data.shape[1] - window_samples + 1, step_samples):
    segment = data[:, start : start + window_samples]

    # Welch PSD for each channel
    freqs, psd = welch(
        segment,
        fs=sfreq,
        nperseg=window_samples,
        axis=-1,
    )

    # Integrate PSD over each frequency band, then average over channels
    alpha_mask = (freqs >= alpha_band[0]) & (freqs <= alpha_band[1])
    delta_mask = (freqs >= delta_band[0]) & (freqs <= delta_band[1])

    alpha_power.append(np.trapz(psd[:, alpha_mask], x=freqs[alpha_mask], axis=-1).mean())
    delta_power.append(np.trapz(psd[:, delta_mask], x=freqs[delta_mask], axis=-1).mean())

    times.append((start + window_samples / 2) / sfreq)

times = np.asarray(times)
alpha_power = np.asarray(alpha_power)
delta_power = np.asarray(delta_power)

In [ ]:
raw_fragment.plot()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(times, alpha_power)
axes[0].set_ylabel("Alpha power ($V^2$)")
axes[0].set_title(f"Alpha-band power ({alpha_band[0]}–{alpha_band[1]} Hz)")
axes[0].set_ylim([0, 1e-10])
axes[0].grid(True, alpha=0.3)

axes[1].plot(times, delta_power)
axes[1].set_ylabel("Delta power ($V^2$)")
axes[1].set_xlabel("Time from start of video-aligned fragment (s)")
axes[1].set_title(f"Delta-band power ({delta_band[0]}–{delta_band[1]} Hz)")
axes[1].set_ylim([0, 0.4*1e-9])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(video_path.parent / f"{video_path.stem}_alpha_delta_powers.pdf")